In [1]:
import pandas as pd
import numpy as np
from scipy.io import arff

# Load the ARFF dataset
data, meta = arff.loadarff("3year.arff")

# Convert to DataFrame
df = pd.DataFrame(data)

# Decode target column if stored as bytes
df["class"] = df["class"].apply(
    lambda x: int(x.decode()) if isinstance(x, bytes) else int(x)
)

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nClass distribution:")
print(df["class"].value_counts())

print("\nMissing values:")
print(df.isnull().sum().sum())

Dataset shape: (10503, 65)

First 5 rows:


,Attr1,Attr2,Attr3,Attr4,Attr5,Attr6,Attr7,Attr8,Attr9,Attr10,...,Attr56,Attr57,Attr58,Attr59,Attr60,Attr61,Attr62,Attr63,Attr64,class
0,0.174190,0.41299,0.14371,1.3480,-28.9820,0.60383,0.219460,1.1225,1.1961,0.46359,...,0.163960,0.375740,0.83604,0.000007,9.7145,6.2813,84.291,4.3303,4.0341,0
1,0.146240,0.46038,0.28230,1.6294,2.5952,0.00000,0.171850,1.1721,1.6018,0.53962,...,0.027516,0.271000,0.90108,0.000000,5.9882,4.1103,102.190,3.5716,5.9500,0
2,0.000595,0.22612,0.48839,3.1599,84.8740,0.19114,0.004572,2.9881,1.0077,0.67566,...,0.007639,0.000881,0.99236,0.000000,6.7742,3.7922,64.846,5.6287,4.4581,0
3,0.024526,0.43236,0.27546,1.7833,-10.1050,0.56944,0.024526,1.3057,1.0509,0.56453,...,0.048398,0.043445,0.95160,0.142980,4.2286,5.0528,98.783,3.6950,3.4844,0
4,0.188290,0.41504,0.34231,1.9279,-58.2740,0.00000,0.233580,1.4094,1.3393,0.58496,...,0.176480,0.321880,0.82635,0.073039,2.5912,7.0756,100.540,3.6303,4.6375,0



Class distribution:
class
0    10008
1      495
Name: count, dtype: int64

Missing values:
9888


In [2]:
# Separate features and target
X = df.drop("class", axis=1)
y = df["class"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# Check missing values
print("\nMissing values per feature:")
print(X.isnull().sum().sum())

Features shape: (10503, 64)
Target shape: (10503,)

Missing values per feature:
9888


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

# Mean imputation
imputer = SimpleImputer(strategy="mean")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

print("\nAfter imputation:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Missing values in X_train:", np.isnan(X_train).sum())
print("Missing values in X_test:", np.isnan(X_test).sum())

Training set: (8402, 64)
Testing set: (2101, 64)

After imputation:
X_train: (8402, 64)
X_test: (2101, 64)
Missing values in X_train: 0
Missing values in X_test: 0


Model 1 — Logistic Regression

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report
)
import time

# Create Logistic Regression model
logistic_model = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=5000,
    random_state=42
)

# Train
start_time = time.time()
logistic_model.fit(X_train, y_train)
train_time = time.time() - start_time

# Predictions
y_pred = logistic_model.predict(X_test)
y_prob = logistic_model.predict_proba(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("===== Logistic Regression =====")
print(f"Test Accuracy : {accuracy:.6f}")
print(f"Test Log Loss : {loss:.6f}")
print(f"Training Time : {train_time:.4f} seconds")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

===== Logistic Regression =====
Test Accuracy : 0.950976
Test Log Loss : 0.222046
Training Time : 37.1591 seconds

Confusion Matrix:
[[1998    4]
 [  99    0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      2002
           1       0.00      0.00      0.00        99

    accuracy                           0.95      2101
   macro avg       0.48      0.50      0.49      2101
weighted avg       0.91      0.95      0.93      2101



Model 2: SVM

In [7]:
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report
)
import time

# Create SVM model
svm_model = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

# Train
start_time = time.time()
svm_model.fit(X_train, y_train)
train_time = time.time() - start_time

# Predictions
y_pred = svm_model.predict(X_test)
y_prob = svm_model.predict_proba(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("===== SVM =====")
print(f"Test Accuracy : {accuracy:.6f}")
print(f"Test Log Loss : {loss:.6f}")
print(f"Training Time : {train_time:.4f} seconds")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred,zero_division=0))

===== SVM =====
Test Accuracy : 0.952880
Test Log Loss : 0.189984
Training Time : 1.8353 seconds

Confusion Matrix:
[[2002    0]
 [  99    0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      2002
           1       0.00      0.00      0.00        99

    accuracy                           0.95      2101
   macro avg       0.48      0.50      0.49      2101
weighted avg       0.91      0.95      0.93      2101



Model 3 — Neural Network (MLP)

In [8]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report
)
import time

# Create Neural Network
mlp_model = MLPClassifier(
    hidden_layer_sizes=(5, 5),
    activation="logistic",
    solver="sgd",
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=1000,
    random_state=42
)

# Train
start_time = time.time()

mlp_model.fit(X_train, y_train)

train_time = time.time() - start_time

# Predictions
y_pred = mlp_model.predict(X_test)
y_prob = mlp_model.predict_proba(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("===== Neural Network (MLP) =====")
print(f"Test Accuracy : {accuracy:.6f}")
print(f"Test Log Loss : {loss:.6f}")
print(f"Training Time : {train_time:.4f} seconds")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

===== Neural Network (MLP) =====
Test Accuracy : 0.952880
Test Log Loss : 0.188515
Training Time : 1.3748 seconds

Confusion Matrix:
[[2002    0]
 [  99    0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      2002
           1       0.00      0.00      0.00        99

    accuracy                           0.95      2101
   macro avg       0.48      0.50      0.49      2101
weighted avg       0.91      0.95      0.93      2101



 Model 4 — Bernoulli Naive Bayes

In [9]:
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report
)
import time

# Create Bernoulli Naive Bayes model
nb_model = BernoulliNB()

# Train
start_time = time.time()

nb_model.fit(X_train, y_train)

train_time = time.time() - start_time

# Predictions
y_pred = nb_model.predict(X_test)
y_prob = nb_model.predict_proba(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("===== Bernoulli Naive Bayes =====")
print(f"Test Accuracy : {accuracy:.6f}")
print(f"Test Log Loss : {loss:.6f}")
print(f"Training Time : {train_time:.4f} seconds")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

===== Bernoulli Naive Bayes =====
Test Accuracy : 0.763922
Test Log Loss : 2.325909
Training Time : 0.0131 seconds

Confusion Matrix:
[[1560  442]
 [  54   45]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.78      0.86      2002
           1       0.09      0.45      0.15        99

    accuracy                           0.76      2101
   macro avg       0.53      0.62      0.51      2101
weighted avg       0.93      0.76      0.83      2101



Model 5 — XGBoost

In [12]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report
)
import time

# Create XGBoost model
xgb_model = XGBClassifier(
    max_depth=3,
    learning_rate=0.001,
    n_estimators=100,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

# Train
start_time = time.time()

xgb_model.fit(X_train, y_train)

train_time = time.time() - start_time

# Predictions
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("===== XGBoost =====")
print(f"Test Accuracy : {accuracy:.6f}")
print(f"Test Log Loss : {loss:.6f}")
print(f"Training Time : {train_time:.4f} seconds")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

===== XGBoost =====
Test Accuracy : 0.952880
Test Log Loss : 0.177626
Training Time : 2.1739 seconds

Confusion Matrix:
[[2002    0]
 [  99    0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      2002
           1       0.00      0.00      0.00        99

    accuracy                           0.95      2101
   macro avg       0.48      0.50      0.49      2101
weighted avg       0.91      0.95      0.93      2101



Model 6 -LightGBM

In [14]:
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report
)
import time

# Create LightGBM model
lgb_model = LGBMClassifier(
    max_depth=3,
    learning_rate=0.001,
    n_estimators=100,
    reg_lambda=1,
    objective="binary",
    random_state=42,
    verbosity=-1
)

# Train
start_time = time.time()

lgb_model.fit(X_train, y_train)

train_time = time.time() - start_time

# Predictions
y_pred = lgb_model.predict(X_test)
y_prob = lgb_model.predict_proba(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("===== LightGBM =====")
print(f"Test Accuracy : {accuracy:.6f}")
print(f"Test Log Loss : {loss:.6f}")
print(f"Training Time : {train_time:.4f} seconds")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

===== LightGBM =====
Test Accuracy : 0.952880
Test Log Loss : 0.177607
Training Time : 3.5655 seconds

Confusion Matrix:
[[2002    0]
 [  99    0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      2002
           1       0.00      0.00      0.00        99

    accuracy                           0.95      2101
   macro avg       0.48      0.50      0.49      2101
weighted avg       0.91      0.95      0.93      2101



Model 7 — Random Forest

In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report
)
import time

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

# Train
start_time = time.time()

rf_model.fit(X_train, y_train)

train_time = time.time() - start_time

# Predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
loss = log_loss(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("===== Random Forest =====")
print(f"Test Accuracy : {accuracy:.6f}")
print(f"Test Log Loss : {loss:.6f}")
print(f"Training Time : {train_time:.4f} seconds")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

===== Random Forest =====
Test Accuracy : 0.958591
Test Log Loss : 0.190344
Training Time : 0.8599 seconds

Confusion Matrix:
[[1996    6]
 [  81   18]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      2002
           1       0.75      0.18      0.29        99

    accuracy                           0.96      2101
   macro avg       0.86      0.59      0.64      2101
weighted avg       0.95      0.96      0.95      2101



5-fold cross-validation for all 7 models.

In [17]:
# 5-FOLD CROSS-VALIDATION FOR ALL 7 MODELS

import time
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier



# 1. DEFINE ALL 7 MODELS

models = {

    "Logistic Regression": LogisticRegression(
        penalty="l2",
        solver="liblinear",
        max_iter=20000,
        random_state=42
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "Neural Network": MLPClassifier(
        hidden_layer_sizes=(5, 5),
        activation="logistic",
        solver="sgd",
        alpha=0.0001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ),

    "Bernoulli Naive Bayes": BernoulliNB(),

    "XGBoost": XGBClassifier(
        max_depth=3,
        learning_rate=0.001,
        n_estimators=100,
        reg_lambda=1,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        max_depth=3,
        learning_rate=0.001,
        n_estimators=100,
        reg_lambda=1,
        objective="binary",
        random_state=42,
        verbosity=-1
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )
}


# 2. 5-FOLD STRATIFIED CROSS-VALIDATION

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# 3. EVALUATION METRICS

scoring = {
    "accuracy": "accuracy",
    "neg_log_loss": "neg_log_loss"
}


# 4. RUN 5-FOLD CROSS-VALIDATION

results = []

for name, model in models.items():

    print(f"\nRunning: {name}")

    # Mean imputation is performed separately
    # inside each fold to avoid data leakage
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("model", model)
    ])

    start_time = time.time()

    cv_results = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1
    )

    total_time = time.time() - start_time

    # Mean accuracy
    mean_accuracy = cv_results["test_accuracy"].mean()
    std_accuracy = cv_results["test_accuracy"].std()

    # Convert negative log loss to positive log loss
    mean_log_loss = -cv_results["test_neg_log_loss"].mean()
    std_log_loss = cv_results["test_neg_log_loss"].std()

    results.append({
        "Model": name,
        "CV Accuracy": mean_accuracy,
        "Accuracy Std": std_accuracy,
        "CV Log Loss": mean_log_loss,
        "Log Loss Std": std_log_loss,
        "CV Time (s)": total_time
    })

    print(f"CV Accuracy : {mean_accuracy:.6f}")
    print(f"CV Log Loss : {mean_log_loss:.6f}")
    print(f"CV Time     : {total_time:.4f} seconds")


# 5. CREATE FINAL RESULTS TABLE

cv_results_df = pd.DataFrame(results)


# 6. DISPLAY FINAL RESULTS

print("\n" + "=" * 80)
print("FINAL 5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 80)

print(
    cv_results_df.to_string(
        index=False,
        formatters={
            "CV Accuracy": "{:.6f}".format,
            "Accuracy Std": "{:.6f}".format,
            "CV Log Loss": "{:.6f}".format,
            "Log Loss Std": "{:.6f}".format,
            "CV Time (s)": "{:.4f}".format
        }
    )
)


Running: Logistic Regression
CV Accuracy : 0.951157
CV Log Loss : 0.219353
CV Time     : 211.4506 seconds

Running: SVM
CV Accuracy : 0.952871
CV Log Loss : 0.190096
CV Time     : 9.8450 seconds

Running: Neural Network
CV Accuracy : 0.952871
CV Log Loss : 0.188357
CV Time     : 4.5611 seconds

Running: Bernoulli Naive Bayes
CV Accuracy : 0.764355
CV Log Loss : 2.441999
CV Time     : 0.1621 seconds

Running: XGBoost
CV Accuracy : 0.952871
CV Log Loss : 0.174666
CV Time     : 0.8074 seconds

Running: LightGBM
CV Accuracy : 0.952871
CV Log Loss : 0.174452
CV Time     : 0.5585 seconds

Running: Random Forest
CV Accuracy : 0.960297
CV Log Loss : 0.170650
CV Time     : 4.1604 seconds

FINAL 5-FOLD CROSS-VALIDATION RESULTS
                Model CV Accuracy Accuracy Std CV Log Loss Log Loss Std CV Time (s)
  Logistic Regression    0.951157     0.001468    0.219353     0.017976    211.4506
                  SVM    0.952871     0.000011    0.190096     0.000145      9.8450
       Neural Networ

 CONSOLIDATED RESULTS TABLE

In [19]:

# FINAL CONSOLIDATED RESULTS TABLE

import pandas as pd

# 1. 80/20 TEST RESULTS

test_results = {
    "Logistic Regression": {
        "Test Accuracy": 0.950976,
        "Test Log Loss": 0.222046,
        "Test Time (s)": 37.1591,
        "Bankrupt Precision": 0.00,
        "Bankrupt Recall": 0.00,
        "Bankrupt F1": 0.00
    },

    "SVM": {
        "Test Accuracy": 0.952880,
        "Test Log Loss": 0.189984,
        "Test Time (s)": 1.8353,
        "Bankrupt Precision": 0.00,
        "Bankrupt Recall": 0.00,
        "Bankrupt F1": 0.00
    },

    "Neural Network": {
        "Test Accuracy": 0.952880,
        "Test Log Loss": 0.188515,
        "Test Time (s)": 1.3748,
        "Bankrupt Precision": 0.00,
        "Bankrupt Recall": 0.00,
        "Bankrupt F1": 0.00
    },

    "Bernoulli Naive Bayes": {
        "Test Accuracy": 0.763922,
        "Test Log Loss": 2.325909,
        "Test Time (s)": 0.0131,
        "Bankrupt Precision": 0.09,
        "Bankrupt Recall": 0.45,
        "Bankrupt F1": 0.15
    },

    "XGBoost": {
        "Test Accuracy": 0.952880,
        "Test Log Loss": 0.177626,
        "Test Time (s)": 2.1739,
        "Bankrupt Precision": 0.00,
        "Bankrupt Recall": 0.00,
        "Bankrupt F1": 0.00
    },

    "LightGBM": {
        "Test Accuracy": 0.952880,
        "Test Log Loss": 0.177607,
        "Test Time (s)": 3.5655,
        "Bankrupt Precision": 0.00,
        "Bankrupt Recall": 0.00,
        "Bankrupt F1": 0.00
    },

    "Random Forest": {
        "Test Accuracy": 0.958591,
        "Test Log Loss": 0.190344,
        "Test Time (s)": 0.8599,
        "Bankrupt Precision": 0.75,
        "Bankrupt Recall": 0.18,
        "Bankrupt F1": 0.29
    }
}


# 2. 5-FOLD CROSS-VALIDATION RESULTS

cv_results = {
    "Logistic Regression": {
        "CV Accuracy": 0.951157,
        "CV Log Loss": 0.219353
    },

    "SVM": {
        "CV Accuracy": 0.952871,
        "CV Log Loss": 0.190096
    },

    "Neural Network": {
        "CV Accuracy": 0.952871,
        "CV Log Loss": 0.188357
    },

    "Bernoulli Naive Bayes": {
        "CV Accuracy": 0.764355,
        "CV Log Loss": 2.441999
    },

    "XGBoost": {
        "CV Accuracy": 0.952871,
        "CV Log Loss": 0.174666
    },

    "LightGBM": {
        "CV Accuracy": 0.952871,
        "CV Log Loss": 0.174452
    },

    "Random Forest": {
        "CV Accuracy": 0.960297,
        "CV Log Loss": 0.170650
    }
}

# 3. COMBINE EVERYTHING

final_results = []

for model in test_results:

    row = {
        "Model": model,

        "Test Accuracy": test_results[model]["Test Accuracy"],
        "Test Log Loss": test_results[model]["Test Log Loss"],
        "Test Time (s)": test_results[model]["Test Time (s)"],

        "CV Accuracy": cv_results[model]["CV Accuracy"],
        "CV Log Loss": cv_results[model]["CV Log Loss"],

        "Bankrupt Precision": test_results[model]["Bankrupt Precision"],
        "Bankrupt Recall": test_results[model]["Bankrupt Recall"],
        "Bankrupt F1": test_results[model]["Bankrupt F1"]
    }

    final_results.append(row)


final_results_df = pd.DataFrame(final_results)


# 4. DISPLAY FINAL TABLE

print("\n" + "=" * 120)
print("FINAL MODEL COMPARISON")
print("=" * 120)

print(
    final_results_df.to_string(
        index=False,
        formatters={
            "Test Accuracy": "{:.6f}".format,
            "Test Log Loss": "{:.6f}".format,
            "Test Time (s)": "{:.4f}".format,
            "CV Accuracy": "{:.6f}".format,
            "CV Log Loss": "{:.6f}".format,
            "Bankrupt Precision": "{:.2f}".format,
            "Bankrupt Recall": "{:.2f}".format,
            "Bankrupt F1": "{:.2f}".format
        }
    )
)
# Save final baseline results
final_results_df.to_csv(
    "final_model_results.csv",
    index=False
)

print("Saved: final_model_results.csv")


FINAL MODEL COMPARISON
                Model Test Accuracy Test Log Loss Test Time (s) CV Accuracy CV Log Loss Bankrupt Precision Bankrupt Recall Bankrupt F1
  Logistic Regression      0.950976      0.222046       37.1591    0.951157    0.219353               0.00            0.00        0.00
                  SVM      0.952880      0.189984        1.8353    0.952871    0.190096               0.00            0.00        0.00
       Neural Network      0.952880      0.188515        1.3748    0.952871    0.188357               0.00            0.00        0.00
Bernoulli Naive Bayes      0.763922      2.325909        0.0131    0.764355    2.441999               0.09            0.45        0.15
              XGBoost      0.952880      0.177626        2.1739    0.952871    0.174666               0.00            0.00        0.00
             LightGBM      0.952880      0.177607        3.5655    0.952871    0.174452               0.00            0.00        0.00
        Random Forest      0.95

Balanced Random Forest

In [20]:
# BALANCED RANDOM FOREST

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import time

balanced_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

balanced_rf.fit(X_train, y_train)

train_time = time.time() - start_time

y_pred_balanced = balanced_rf.predict(X_test)
y_prob_balanced = balanced_rf.predict_proba(X_test)

accuracy_balanced = accuracy_score(y_test, y_pred_balanced)
logloss_balanced = log_loss(y_test, y_prob_balanced)

precision_balanced = precision_score(
    y_test, y_pred_balanced, zero_division=0
)

recall_balanced = recall_score(
    y_test, y_pred_balanced, zero_division=0
)

f1_balanced = f1_score(
    y_test, y_pred_balanced, zero_division=0
)

cm_balanced = confusion_matrix(
    y_test, y_pred_balanced
)

print("\n" + "=" * 70)
print("BALANCED RANDOM FOREST")
print("=" * 70)

print(f"Test Accuracy : {accuracy_balanced:.6f}")
print(f"Test Log Loss : {logloss_balanced:.6f}")
print(f"Training Time : {train_time:.4f} seconds")

print(f"\nBankrupt Precision : {precision_balanced:.4f}")
print(f"Bankrupt Recall    : {recall_balanced:.4f}")
print(f"Bankrupt F1        : {f1_balanced:.4f}")

print("\nConfusion Matrix:")
print(cm_balanced)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_balanced,
        zero_division=0
    )
)


BALANCED RANDOM FOREST
Test Accuracy : 0.959543
Test Log Loss : 0.189176
Training Time : 0.6000 seconds

Bankrupt Precision : 0.8889
Bankrupt Recall    : 0.1616
Bankrupt F1        : 0.2735

Confusion Matrix:
[[2000    2]
 [  83   16]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98      2002
           1       0.89      0.16      0.27        99

    accuracy                           0.96      2101
   macro avg       0.92      0.58      0.63      2101
weighted avg       0.96      0.96      0.95      2101



In [21]:
# SAVE BALANCED RANDOM FOREST RESULTS

balanced_rf_results = pd.DataFrame([{
    "Model": "Balanced Random Forest",
    "Test Accuracy": accuracy_balanced,
    "Test Log Loss": logloss_balanced,
    "Training Time": train_time,
    "Bankrupt Precision": precision_balanced,
    "Bankrupt Recall": recall_balanced,
    "Bankrupt F1": f1_balanced
}])

balanced_rf_results.to_csv(
    "balanced_random_forest_results.csv",
    index=False
)

print("Balanced Random Forest results saved.")

Balanced Random Forest results saved.


done
